# GLDS-525 Pipeline, Notebook 2: Genome Indexing & Alignment

**Study:** RR-23 Mouse Cerebellum Bulk RNA-Seq (NASA GeneLab OSD-525 / GLDS-525)  
**Genome:** Ensembl GRCm39 release 112 + ERCC92 spike-ins

---

## 🎓 Tutorial Introduction: What is Genome Alignment?

In Notebook 1, we cleaned our raw sequencing reads. Now we need to figure out
**where in the mouse genome each read came from, a process called **alignment** (or mapping).

### Why do we need a reference genome?
Imagine you've shredded a 25,000-page book (the genome) into confetti (the reads).
To reconstruct the book, you need an intact copy as a reference to figure out where
each piece belongs. The **reference genome** (GRCm39, the current mouse genome assembly)
is that intact copy.

### What are ERCC92 spike-ins?
**ERCC92** are 92 synthetic RNA molecules of known sequence and concentration
that were added to each sample before sequencing. Because we know exactly how much
of each spike-in was added, they act as internal controls, helping normalize
differences in library preparation between samples.

### What is the GTF file?
The **GTF (Gene Transfer Format)** file is an annotation file that tells us where
each gene is located in the genome. Without it, we could align reads to the genome
but wouldn't know which gene they belong to.

---

## What This Notebook Does

| Step | Description | Why It Matters |
|---|---|---|
| **1** | Load config from Notebook 1 | Restores all paths and settings |
| **2** | Download reference genome + GTF + ERCC92 | Gets the ~1.6 GB reference files |
| **3** | Merge genome FASTA + ERCC92 | Combines mouse genome with spike-in sequences |
| **4** | Build STAR genome index | One-time index (~1–2 hours, ~32 GB RAM) |
| **5** | Align all samples with STAR | Maps reads to genome (~30–60 min per sample) |
| **6** | Sort & index BAM files | Prepares alignments for downstream tools |
| **7** | Post-alignment QC with RSeQC | Verifies alignment quality |
| **8** | Alignment MultiQC report | Aggregates alignment statistics |

> **⚠️ RAM requirement:** STAR genome indexing requires approximately **32 GB RAM**.
> Alignment requires ~32 GB RAM per job. Check your available RAM before proceeding.


---
## Section 1: Load Configuration from Notebook 1

> **Overview:** In Notebook 1, we saved all our directory paths and
> system settings to a JSON config file. Loading it here means we don't have to
> re-enter any paths; the pipeline "remembers" where everything is.
>
> If this cell raises a `FileNotFoundError`, either run Notebook 1 first, or
> update `CONFIG_PATH` to point to your `pipeline_config.json`.


In [ ]:
import json
import subprocess
import os
import time
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing

# ── Load config saved by Notebook 1 ──────────────────────────────────────────
# ← EDIT if you moved the project folder
CONFIG_PATH = Path.home() / 'GLDS525_project' / 'pipeline_config.json'

if not CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"Config not found: {CONFIG_PATH}\n"
        "Please run Notebook 1 first, or update CONFIG_PATH above."
    )

with open(CONFIG_PATH) as f:
    config = json.load(f)

# Restore path variables
PROJECT_ROOT   = Path(config['PROJECT_ROOT'])
N_CORES        = config['N_CORES']
TOTAL_RAM_GB   = config['TOTAL_RAM_GB']
DIRS           = {k: Path(v) for k, v in config.items()
                  if k not in ('PROJECT_ROOT', 'N_CORES', 'TOTAL_RAM_GB')}

print(f"✅ Config loaded from: {CONFIG_PATH}")
print(f"   Project root: {PROJECT_ROOT}")
print(f"   Usable cores: {N_CORES}")
print(f"   Total RAM:    {TOTAL_RAM_GB} GB")


---
## Section 2: Download Reference Genome, GTF, and ERCC92

> **Overview:** We need three reference files:
>
> 1. **Genome FASTA** (`Mus_musculus.GRCm39.dna.primary_assembly.fa`): The actual
>    DNA sequence of all mouse chromosomes, ~800 MB compressed. This is what
>    STAR will align our reads against.
>
> 2. **GTF annotation** (`Mus_musculus.GRCm39.112.gtf`): A table listing the
>    coordinates of every known gene and transcript, ~800 MB compressed. STAR
>    uses this to know where gene boundaries are so it can correctly handle
>    reads spanning exon-exon junctions (splice junctions).
>
> 3. **ERCC92 sequences**: The 92 spike-in RNA sequences. We'll merge these
>    with the mouse genome so STAR can also align spike-in reads.
>
> We use **Ensembl release 112**, the same version GeneLab used. Using a different
> version would change gene IDs and make our results incomparable.


In [ ]:
# Reference file URLs for Ensembl GRCm39 release 112 (exact version used by GeneLab)
ENSEMBL_BASE = "https://ftp.ensembl.org/pub/release-112/"

REFERENCES = {
    'genome_fa_gz': {
        'url':   ENSEMBL_BASE + "fasta/mus_musculus/dna/Mus_musculus.GRCm39.dna.primary_assembly.fa.gz",
        'local': DIRS['genome'] / 'Mus_musculus.GRCm39.dna.primary_assembly.fa.gz',
        'final': DIRS['genome'] / 'Mus_musculus.GRCm39.dna.primary_assembly.fa',
        'size_hint': '800 MB'
    },
    'gtf_gz': {
        'url':   ENSEMBL_BASE + "gtf/mus_musculus/Mus_musculus.GRCm39.112.gtf.gz",
        'local': DIRS['genome'] / 'Mus_musculus.GRCm39.112.gtf.gz',
        'final': DIRS['genome'] / 'Mus_musculus.GRCm39.112.gtf',
        'size_hint': '800 MB'
    },
    'ercc92_fa': {
        'url':   "https://assets.thermofisher.com/TFS-Assets/LSG/manuals/ERCC92.zip",
        'local': DIRS['genome'] / 'ERCC92.zip',
        'final': DIRS['genome'] / 'ERCC92.fa',
        'size_hint': '250 KB'
    },
    'ercc92_gtf': {
        'url':   "https://assets.thermofisher.com/TFS-Assets/LSG/manuals/ERCC92.zip",
        'local': DIRS['genome'] / 'ERCC92.zip',
        'final': DIRS['genome'] / 'ERCC92.gtf',
        'size_hint': '250 KB (same zip as FASTA)'
    }
}

for name, ref in REFERENCES.items():
    exists = ref['final'].exists()
    status = '✅ exists' if exists else '⬇  needed'
    print(f"  {status}  {ref['final'].name}  ({ref['size_hint']})")


In [ ]:
def wget_download(url, out_path, desc=''):
    """Download a file using wget with resume support."""
    if out_path.exists():
        print(f"  ⏭  SKIP  {out_path.name} (already exists)")
        return True
    print(f"  ⬇  Downloading {desc or out_path.name}...")
    cmd = ['wget', '--continue', '--tries=5', '--timeout=120',
           '--output-document', str(out_path), url]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode == 0:
        print(f"  ✅ {out_path.name}")
        return True
    else:
        print(f"  ❌ Failed: {result.stderr[:200]}")
        return False

def gunzip_file(gz_path, out_path):
    """Decompress a .gz file."""
    if out_path.exists():
        print(f"  ⏭  SKIP  {out_path.name} (already decompressed)")
        return True
    print(f"  Decompressing {gz_path.name}...")
    result = subprocess.run(['gunzip', '-k', str(gz_path)], capture_output=True, text=True)
    if result.returncode == 0:
        print(f"  ✅ {out_path.name}")
        return True
    else:
        print(f"  ❌ {result.stderr[:200]}")
        return False

# Download genome FASTA
ref = REFERENCES['genome_fa_gz']
wget_download(ref['url'], ref['local'], 'GRCm39 primary assembly FASTA (~800 MB)')
gunzip_file(ref['local'], ref['final'])

# Download GTF
ref = REFERENCES['gtf_gz']
wget_download(ref['url'], ref['local'], 'GRCm39 release 112 GTF (~800 MB)')
gunzip_file(ref['local'], ref['final'])

# Download ERCC92 zip
ercc_zip = DIRS['genome'] / 'ERCC92.zip'
ercc_fa  = DIRS['genome'] / 'ERCC92.fa'
ercc_gtf = DIRS['genome'] / 'ERCC92.gtf'
wget_download(
    "https://assets.thermofisher.com/TFS-Assets/LSG/manuals/ERCC92.zip",
    ercc_zip, 'ERCC92 spike-in sequences'
)
if ercc_zip.exists() and (not ercc_fa.exists() or not ercc_gtf.exists()):
    print("  Extracting ERCC92.zip...")
    subprocess.run(['unzip', '-o', str(ercc_zip), '-d', str(DIRS['genome'])],
                   capture_output=True)
    print(f"  ✅ ERCC92.fa and ERCC92.gtf extracted")


---
## Section 3: Merge Genome + ERCC92 References

> **Overview:** To align reads from both the mouse genome and the
> ERCC92 spike-ins in a single pass, we combine them into one big FASTA file
> and one big GTF file. This is exactly what GeneLab did.
>
> Think of it as appending 92 extra "chromosomes" (the spike-in sequences) to
> the end of the mouse genome file. STAR will then be able to align reads that
> came from either source.
>
> We also extract a list of **rRNA gene IDs** from the GTF; these are ribosomal RNA genes
> that we'll filter out in Notebook 3, because rRNA contamination can distort
> normalization.


In [ ]:
genome_fa  = DIRS['genome'] / 'Mus_musculus.GRCm39.dna.primary_assembly.fa'
genome_gtf = DIRS['genome'] / 'Mus_musculus.GRCm39.112.gtf'
ercc_fa    = DIRS['genome'] / 'ERCC92.fa'
ercc_gtf   = DIRS['genome'] / 'ERCC92.gtf'

merged_fa  = DIRS['genome'] / 'Mus_musculus.GRCm39.dna.primary_assembly_and_ERCC92.fa'
merged_gtf = DIRS['genome'] / 'Mus_musculus.GRCm39.112_and_ERCC92.gtf'

if not merged_fa.exists():
    print("Merging genome FASTA + ERCC92 FASTA...")
    with open(merged_fa, 'wb') as out:
        for src in [genome_fa, ercc_fa]:
            with open(src, 'rb') as f:
                out.write(f.read())
    size_gb = merged_fa.stat().st_size / 1e9
    print(f"  ✅ {merged_fa.name}  ({size_gb:.2f} GB)")
else:
    print(f"  ⏭  SKIP  {merged_fa.name}")

if not merged_gtf.exists():
    print("Merging GTF + ERCC92 GTF...")
    with open(merged_gtf, 'wb') as out:
        for src in [genome_gtf, ercc_gtf]:
            with open(src, 'rb') as f:
                out.write(f.read())
    size_mb = merged_gtf.stat().st_size / 1e6
    print(f"  ✅ {merged_gtf.name}  ({size_mb:.1f} MB)")
else:
    print(f"  ⏭  SKIP  {merged_gtf.name}")

# Extract rRNA gene IDs (used later for rRNA removal in counts)
rrna_ids_file = DIRS['genome'] / 'mus_musculus_rrna_ids.txt'
if not rrna_ids_file.exists():
    print("Extracting rRNA gene IDs from GTF...")
    cmd = (
        f"grep -E 'gene_biotype \"rRNA\"|gene_type \"rRNA\"|gbkey \"rRNA\"' {merged_gtf} "
        f"| grep -o 'gene_id \"[^\"]*\"' "
        f"| sed 's/gene_id \"\\(.*\\)\"/\\1/' "
        f"| sort -u > {rrna_ids_file}"
    )
    subprocess.run(cmd, shell=True)
    n_rrna = sum(1 for _ in open(rrna_ids_file))
    print(f"  ✅ Found {n_rrna} rRNA gene IDs → {rrna_ids_file.name}")
else:
    print(f"  ⏭  rRNA IDs already extracted")


---
## Section 4: Build STAR Genome Index

> **Overview:** Before STAR can align reads, it needs to build a
> special index of the genome, similar to a book's index, but for DNA sequences.
> This index allows STAR to quickly look up where a read might belong without
> scanning the entire 2.5 billion base-pair genome for every read.
>
> **This step:**
> - Takes **1–2 hours** to complete
> - Requires **~32 GB of RAM** during indexing
> - Only needs to be done **once**; the index is reused for all 29 samples
>
> Key parameters:
> - `--sjdbOverhang 150`: read length minus 1 (our reads are 151 bp), optimizes
>   detection of splice junctions
> - `--genomeSAindexNbases`: automatically calculated from genome size

> **⚠️ If your machine has less than 32 GB RAM:** This step will fail. Consider
> running on a cloud computing instance (e.g., AWS, Google Cloud) with sufficient memory.


In [ ]:
import math

STAR_INDEX_DIR = DIRS['star_index'] / 'Mus_musculus.GRCm39.dna.primary_assembly_and_ERCC92_RL-151'
STAR_INDEX_DIR.mkdir(parents=True, exist_ok=True)

# Check if index already built (presence of SA file indicates completion)
if (STAR_INDEX_DIR / 'SA').exists():
    print(f"✅ STAR index already built: {STAR_INDEX_DIR}")
else:
    print("Computing genomeSAindexNbases from genome size...")
    result = subprocess.run(
        f"grep -v '>' {merged_fa} | wc -c",
        shell=True, capture_output=True, text=True
    )
    lines_result = subprocess.run(
        f"grep -v '>' {merged_fa} | wc -l",
        shell=True, capture_output=True, text=True
    )
    num_chars = int(result.stdout.strip())
    num_lines = int(lines_result.stdout.strip())
    num_bases = num_chars - num_lines   # subtract newline characters
    
    sa_index = min(14, int(math.log2(num_bases) / 2 - 1))
    print(f"  Genome bases:       {num_bases:,}")
    print(f"  genomeSAindexNbases: {sa_index}  (min(14, log2(N)/2 - 1))")
    
    # RAM limit: use 90% of available RAM
    ram_limit = int(TOTAL_RAM_GB * 0.90 * 1024**3)
    print(f"  RAM limit:           {ram_limit / 1e9:.1f} GB")
    
    cmd = [
        'STAR',
        '--runThreadN',              str(N_CORES),
        '--runMode',                 'genomeGenerate',
        '--limitGenomeGenerateRAM',  str(ram_limit),
        '--genomeSAindexNbases',     str(sa_index),
        '--genomeDir',               str(STAR_INDEX_DIR),
        '--genomeFastaFiles',        str(merged_fa),
        '--sjdbGTFfile',             str(merged_gtf),
        '--sjdbOverhang',            '150'
    ]
    
    print(f"\nBuilding STAR index with {N_CORES} threads...")
    print("This will take 1–2 hours. Log → logs/star_genome_generate.log")
    
    t0 = time.time()
    result = subprocess.run(cmd, capture_output=True, text=True)
    elapsed = (time.time() - t0) / 60
    
    log_path = DIRS['logs'] / 'star_genome_generate.log'
    log_path.write_text(result.stdout + result.stderr)
    
    if result.returncode == 0:
        print(f"\n✅ STAR index built successfully ({elapsed:.1f} minutes)")
    else:
        print(f"\n❌ STAR index build failed. Check: {log_path}")
        print(result.stderr[-500:])


---
## Section 5: Align All Samples with STAR

> **Overview:** Now we align the trimmed reads from each of our 29 samples
> to the reference genome. **STAR (Spliced Transcripts Alignment to a Reference)** is
> designed specifically for RNA-seq because it handles **spliced reads** (reads that span across introns
> non-coding regionsions that are spliced out when DNA becomes mRNA).
>
> For each sample, STAR produces:
> - **Coordinate-sorted BAM** (`Aligned.sortedByCoord.out.bam`): reads ordered by
>   their position in the genome, used for visualization and RSeQC QC
> - **Transcriptome BAM** (`Aligned.toTranscriptome.out.bam`): reads aligned to
>   transcript coordinates, required as input for RSEM quantification (Notebook 3)
> - **Splice junction file** (`SJ.out.tab`): a table of all detected splice junctions
>
> **Time estimate:** Each sample takes ~30–60 minutes on a 16-core workstation.
> With 29 samples, plan for this to run overnight.

**GeneLab parameters reproduced** (from processing info):
- Two-pass alignment via pre-built index with splice junction database
- Output: `Aligned.sortedByCoord.out.bam` and `Aligned.toTranscriptome.out.bam`


In [ ]:
import pandas as pd

# Load runsheet to get sample list
RUNSHEET_PATH = DIRS['runsheet'] / 'GLDS-525_rna_seq_bulkRNASeq_v2_runsheet.csv'
runsheet = pd.read_csv(RUNSHEET_PATH)

def get_trimmed_pair(sample, trimmed_dir):
    r1 = trimmed_dir / f"{sample}_GLbulkRNAseq_R1_trimmed.fastq.gz"
    r2 = trimmed_dir / f"{sample}_GLbulkRNAseq_R2_trimmed.fastq.gz"
    return r1, r2

def star_done(sample, aligned_dir):
    """Check if STAR alignment finished for this sample."""
    bam = aligned_dir / f"{sample}_GLbulkRNAseq_Aligned.sortedByCoord.out.bam"
    return bam.exists() and bam.stat().st_size > 1_000_000

samples = runsheet['Sample Name'].tolist()
todo_align = [s for s in samples if not star_done(s, DIRS['aligned'])]
print(f"Samples total:       {len(samples)}")
print(f"Already aligned:     {len(samples) - len(todo_align)}")
print(f"To align:            {len(todo_align)}")


In [ ]:
def run_star_alignment(sample, trimmed_dir, aligned_dir, star_index, logs_dir,
                       threads=8, merged_gtf=None):
    """
    Align one sample with STAR. Produces coordinate-sorted BAM and
    transcriptome BAM (for RSEM). Output files renamed to GeneLab convention.
    """
    r1, r2 = get_trimmed_pair(sample, trimmed_dir)
    if not r1.exists() or not r2.exists():
        return False, sample, f"Trimmed FASTQs not found"
    
    out_prefix = str(aligned_dir / f"{sample}_GLbulkRNAseq_")
    
    cmd = [
        'STAR',
        '--runThreadN',                  str(threads),
        '--genomeDir',                   str(star_index),
        '--sjdbGTFfile',                 str(merged_gtf),
        '--readFilesIn',                 str(r1), str(r2),
        '--readFilesCommand',            'zcat',
        '--outSAMtype',                  'BAM', 'SortedByCoordinate',
        '--outSAMattributes',            'NH', 'HI', 'AS', 'NM', 'MD',
        '--outSAMunmapped',              'Within',
        '--quantMode',                   'TranscriptomeSAM', 'GeneCounts',
        '--outFileNamePrefix',           out_prefix,
        '--outFilterType',               'BySJout',
        '--outFilterMultimapNmax',       '20',
        '--alignSJoverhangMin',          '8',
        '--alignSJDBoverhangMin',        '1',
        '--outFilterMismatchNmax',       '999',
        '--outFilterMismatchNoverLmax',  '0.1',
        '--alignIntronMin',              '20',
        '--alignIntronMax',              '1000000',
        '--alignMatesGapMax',            '1000000',
        '--limitBAMsortRAM',             str(int(TOTAL_RAM_GB * 0.3 * 1e9)),
    ]
    
    t0 = time.time()
    result = subprocess.run(cmd, capture_output=True, text=True)
    elapsed = (time.time() - t0) / 60
    
    log_path = logs_dir / f"star_{sample}.log"
    log_path.write_text(result.stdout + result.stderr)
    
    if result.returncode != 0:
        return False, sample, result.stderr[-200:]
    
    # Move STAR log to logs directory
    rename_map = {
        f"{out_prefix}Log.final.out": str(logs_dir / f"{sample}_GLbulkRNAseq_Log.final.out"),
    }
    for src, dst in rename_map.items():
        src_path = Path(src)
        if src_path.exists() and src != dst:
            src_path.rename(dst)
    
    return True, sample, f"{elapsed:.1f} min"


if todo_align:
    # STAR is memory-intensive; run one job at a time per available RAM
    star_parallel = max(1, int(TOTAL_RAM_GB / 32))
    star_threads  = max(4, N_CORES // star_parallel)
    print(f"Aligning {len(todo_align)} samples:")
    print(f"  Parallel STAR jobs: {star_parallel}  |  Threads per job: {star_threads}")
    print("  Each sample takes ~30–60 min on a 16-core workstation.")
    
    failed = []
    with ProcessPoolExecutor(max_workers=star_parallel) as ex:
        futures = [
            ex.submit(run_star_alignment, s, DIRS['trimmed_fastq'],
                      DIRS['aligned'], STAR_INDEX_DIR, DIRS['logs'],
                      star_threads, merged_gtf)
            for s in todo_align
        ]
        for i, future in enumerate(as_completed(futures), 1):
            ok, sample, msg = future.result()
            status = '✅' if ok else '❌'
            print(f"  [{i:02d}/{len(todo_align)}] {status} {sample}  {msg}")
            if not ok:
                failed.append((sample, msg))
    
    if failed:
        print(f"\n❌ Failed: {failed}")
    else:
        print(f"\n✅ STAR alignment complete for all samples.")
else:
    print("✅ All samples already aligned.")


---
## Section 6: Sort & Index BAM Files with SAMtools

> **Overview:** A **BAM file** is a compressed binary file containing
> all the alignment information, including which read aligned where, with what quality, etc.
>
> **Indexing** a BAM file creates a `.bai` index file that allows other tools to
> quickly jump to any position in the genome without reading the entire file from
> the beginning, similar to a database index speeding up queries.
>
> SAMtools handles this efficiently in parallel across all samples.


In [ ]:
def index_bam(bam_path, threads=4):
    """Index a BAM file with samtools. Skip if index already exists."""
    bai_path = Path(str(bam_path) + '.bai')
    if bai_path.exists():
        return True, bam_path.name
    cmd = ['samtools', 'index', '-@', str(threads), str(bam_path)]
    result = subprocess.run(cmd, capture_output=True, text=True)
    return result.returncode == 0, bam_path.name

# Find all coordinate-sorted BAMs
coord_bams = sorted(DIRS['aligned'].glob('*_Aligned.sortedByCoord.out.bam'))
print(f"Found {len(coord_bams)} coordinate-sorted BAM files")

to_index = [b for b in coord_bams if not Path(str(b) + '.bai').exists()]
print(f"Already indexed: {len(coord_bams) - len(to_index)}  |  To index: {len(to_index)}")

if to_index:
    index_workers = max(1, N_CORES // 4)
    print(f"Indexing {len(to_index)} BAMs ({index_workers} parallel, 4 threads each)...")
    
    with ProcessPoolExecutor(max_workers=index_workers) as ex:
        futures = [ex.submit(index_bam, b, 4) for b in to_index]
        for i, future in enumerate(as_completed(futures), 1):
            ok, fname = future.result()
            print(f"  [{i:02d}/{len(to_index)}] {'✅' if ok else '❌'} {fname}")
    print("✅ BAM indexing complete.")
else:
    print("✅ All BAMs already indexed.")


---
## Section 7: Post-Alignment QC with RSeQC

> **Overview:** Even if reads align successfully, alignment quality
> can still reveal problems. **RSeQC** runs several checks on the aligned BAM files:
>
> | Module | What it measures | What to expect |
> |---|---|---|
> | `infer_experiment` | Library strandedness | "sense" for this dataset |
> | `inner_distance` | Insert size between read pairs | ~150–200 bp peak |
> | `read_distribution` | Where reads map (exons, introns, intergenic) | >60% in exons |
> | `geneBody_coverage` | 5' to 3' read coverage across genes | Relatively flat; a steep drop indicates degradation |
>
> **Strandedness** is particularly important: it tells us whether reads come from
> the same strand as the gene (sense/forward) or the opposite strand (antisense/reverse).
> Getting this wrong would reverse which gene the reads are assigned to!
> GeneLab confirmed this library is **sense-stranded (forward)**.
>
> These RSeQC outputs also get included in the MultiQC report in Section 8.

> A BED12 file is required for RSeQC. We generate it from the GTF using
> `gtfToGenePred` and `genePredToBed` (matching GeneLab's approach).


In [ ]:
# Convert GTF → genePred → BED12 (required by RSeQC)
bed_file    = DIRS['genome'] / 'Mus_musculus.GRCm39.112_and_ERCC92.gtf.bed'
genepred    = DIRS['genome'] / 'Mus_musculus.GRCm39.112_and_ERCC92.gtf.genePred'

if not genepred.exists():
    print("Converting GTF → genePred...")
    cmd = [
        'gtfToGenePred',
        '-geneNameAsName2',
        '-ignoreGroupsWithoutExons',
        str(merged_gtf),
        str(genepred)
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(f"  {'✅' if result.returncode == 0 else '❌'} genePred created")
else:
    print("  ⏭  genePred already exists")

if not bed_file.exists():
    print("Converting genePred → BED12...")
    cmd = ['genePredToBed', str(genepred), str(bed_file)]
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(f"  {'✅' if result.returncode == 0 else '❌'} BED12 file created: {bed_file.name}")
else:
    print(f"  ⏭  BED file already exists")


In [ ]:
def run_rseqc_sample(bam_path, bed_file, rseqc_dir, logs_dir):
    """Run all RSeQC modules for one sample."""
    sample = bam_path.name.split('_GLbulkRNAseq_')[0]
    out_prefix = str(rseqc_dir / sample)
    errors = []
    
    modules = {
        'infer_experiment': [
            'infer_experiment.py', '-i', str(bam_path), '-r', str(bed_file)
        ],
        'inner_distance': [
            'inner_distance.py', '-i', str(bam_path), '-r', str(bed_file),
            '-o', f"{out_prefix}_inner_dist"
        ],
        'read_distribution': [
            'read_distribution.py', '-i', str(bam_path), '-r', str(bed_file)
        ],
        'geneBody_coverage': [
            'geneBody_coverage.py', '-i', str(bam_path),
            '-r', str(bed_file), '-o', out_prefix
        ],
    }
    
    for mod_name, cmd in modules.items():
        result = subprocess.run(cmd, capture_output=True, text=True)
        log = logs_dir / f"rseqc_{sample}_{mod_name}.log"
        log.write_text(result.stdout + result.stderr)
        if result.returncode != 0:
            errors.append(mod_name)
    
    return len(errors) == 0, sample, errors


if bed_file.exists():
    rseqc_done_samples = {f.stem.split('_infer')[0] for f in DIRS['rseqc'].glob('*.log')}
    bams_to_qc = [b for b in coord_bams
                  if b.name.split('_GLbulkRNAseq_')[0] not in rseqc_done_samples]
    
    print(f"RSeQC: {len(bams_to_qc)} samples to process")
    
    rseqc_workers = min(8, N_CORES)
    failed = []
    with ProcessPoolExecutor(max_workers=rseqc_workers) as ex:
        futures = [
            ex.submit(run_rseqc_sample, b, bed_file, DIRS['rseqc'], DIRS['logs'])
            for b in bams_to_qc
        ]
        for i, future in enumerate(as_completed(futures), 1):
            ok, sample, errs = future.result()
            status = '✅' if ok else f'⚠️  (failed: {errs})'
            print(f"  [{i:02d}/{len(bams_to_qc)}] {status} {sample}")
            if not ok:
                failed.append(sample)
    
    if not failed:
        print(f"\n✅ RSeQC complete for all samples.")
    else:
        print(f"\n⚠️  Some RSeQC modules failed for: {failed}")
        print("   Check individual logs in the logs/ directory.")
else:
    print("⚠️  BED file not found. Install gtfToGenePred and genePredToBed (UCSC tools)")
    print("   conda install -c bioconda ucsc-gtftogenepred ucsc-genepredtobed")


---
## Section 8: Alignment MultiQC Report

> **Overview:** This final MultiQC report combines the STAR alignment
> statistics and all RSeQC outputs into one interactive HTML file. Key things to
> check before moving on to quantification:
>
> - **Uniquely mapped reads** should be **>80%** for mouse cerebellum tissue.
>   If it's much lower, something went wrong with alignment parameters.
> - **`infer_experiment`** output should confirm **sense** strandedness for all samples.
> - **Gene body coverage** should be relatively flat; a strong 3' bias suggests RNA degradation occurred during sample preparation.
> - **Read distribution** should show most reads in exons/UTRs, not introns or intergenic.


In [ ]:
report_name = 'GLDS-525_rna_seq_align_multiqc_GLbulkRNAseq'

cmd = [
    'multiqc',
    '--force',
    '--interactive',
    '-o', str(DIRS['multiqc_trimmed']),
    '-n', report_name,
    str(DIRS['aligned']),
    str(DIRS['logs']),
    str(DIRS['rseqc'])
]
print("Running alignment MultiQC...")
result = subprocess.run(cmd, capture_output=True, text=True)
(DIRS['logs'] / 'multiqc_alignment.log').write_text(result.stdout + result.stderr)

report_path = DIRS['multiqc_trimmed'] / f"{report_name}.html"
if report_path.exists():
    print(f"✅ Alignment MultiQC report: {report_path}")
    print(f"\n→ Proceed to Notebook 3: RSEM Quantification")
else:
    print(f"❌ MultiQC failed. Check: {DIRS['logs'] / 'multiqc_alignment.log'}")
